In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [2]:
sieve_percentage_df = pd.read_csv(r"C:\Users\LENOVO\Documents\PostGrad\BSc Hons\4ZOL509_Research\data\seive_percentage.csv")

In [4]:
# Define sieve sizes (mm) and phi equivalents
sieve_sizes_mm = np.array([2.0, 1.0, 0.5, 0.25, 0.125, 0.063, 0.0315])
phi_sizes = -np.log2(sieve_sizes_mm)

# Folk & Ward function
def folk_and_ward_stats(phi, perc):
    cum_perc = np.cumsum(perc)
    D5 = np.interp(5, cum_perc, phi)
    D16 = np.interp(16, cum_perc, phi)
    D25 = np.interp(25, cum_perc, phi)
    D50 = np.interp(50, cum_perc, phi)
    D75 = np.interp(75, cum_perc, phi)
    D84 = np.interp(84, cum_perc, phi)
    D95 = np.interp(95, cum_perc, phi)

    mean_phi = (D16 + D50 + D84) / 3
    sorting = (D84 - D16) / 4 + (D95 - D5) / 6.6
    skewness = ((D16 + D84 - 2 * D50) / (2 * (D84 - D16))) + ((D5 + D95 - 2 * D50) / (2 * (D95 - D5)))
    kurtosis = (D95 - D5) / (2.44 * (D75 - D25))

    return {
        "Mean (φ)": round(mean_phi, 2),
        "Sorting (σφ)": round(sorting, 2),
        "Skewness (Skφ)": round(skewness, 2),
        "Kurtosis (Kφ)": round(kurtosis, 2),
        "D10 (φ)": round(np.interp(10, cum_perc, phi), 2),
        "D50 (φ)": round(D50, 2),
        "D90 (φ)": round(np.interp(90, cum_perc, phi), 2)
    }

# Moments in mm
def additional_moments(mm_sizes, perc):
    mm_sizes = np.array(mm_sizes)
    perc = np.array(perc) / np.sum(perc)

    arithmetic_mean = np.sum(mm_sizes * perc)
    geometric_mean = np.exp(np.sum(np.log(mm_sizes) * perc))
    logarithmic_mean = np.exp(np.sum(np.log10(mm_sizes) * perc))

    variance = np.sum(((mm_sizes - arithmetic_mean) ** 2) * perc)
    std_dev = math.sqrt(variance)
    skew = np.sum(((mm_sizes - arithmetic_mean) ** 3) * perc) / std_dev**3
    kurt = np.sum(((mm_sizes - arithmetic_mean) ** 4) * perc) / std_dev**4

    return {
        "Arithmetic Mean (mm)": round(arithmetic_mean, 4),
        "Geometric Mean (mm)": round(geometric_mean, 4),
        "Logarithmic Mean (mm)": round(logarithmic_mean, 4),
        "Std Dev (mm)": round(std_dev, 4),
        "Skewness (mm)": round(skew, 4),
        "Kurtosis (mm)": round(kurt, 4)
    }

# Analyze each replicate
results = []
for _, row in sieve_percentage_df.iterrows():
    site = row['site']
    replicate = row['replicate']
    perc = row.iloc[2:].astype(float).to_numpy()
    perc = perc / np.sum(perc) * 100

    fw_stats = folk_and_ward_stats(phi_sizes, perc)
    mm_stats = additional_moments(sieve_sizes_mm, perc)

    results.append({
        "Site": site,
        "Replicate": replicate,
        **fw_stats,
        **mm_stats
    })

# Convert to DataFrame
final_df = pd.DataFrame(results)